In [1]:
import sys
print("Python executable:", sys.executable)

try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        print("Current GPU:", torch.cuda.get_device_name(0))
        print("Compute capability:", torch.cuda.get_device_capability(0))
    else:
        print("No CUDA-capable GPU detected by PyTorch.")
except ImportError as e:
    print("PyTorch not installed:", e)

Python executable: /home/andreas-olsson/Dev/FRTN65_code/venv/bin/python3
PyTorch version: 2.14.0+cu132
CUDA available: True
GPU count: 1
Current GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Compute capability: (8, 9)


In [6]:
import time
import torch
 
if not torch.cuda.is_available():
    raise SystemExit("CUDA not available - check your driver/PyTorch install.")
 
device = torch.device("cuda")
print("Using device:", torch.cuda.get_device_name(0))
 
# Big enough to actually load the GPU, small enough not to blow past 8GB VRAM
size = 6000
 
 
a = torch.randn(size, size, device=device)
b = torch.randn(size, size, device=device)

stop = 60
log_interval = stop // 3

start = time.time()
iterations = 0
_time = time.time()
_log_time = _time

print(f"Running repeated {size}x{size} matmuls on GPU for ~{stop} seconds...")
print("Watch it with: watch -n 0.5 nvidia-smi   (in another terminal)")

while _time - start < stop:
    if (_log_time) >= log_interval:
        print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
        print(torch.cuda.memory_reserved() / 1e9, "GB reserved")
        print(torch.cuda.max_memory_allocated() / 1e9, "GB peak")
        _log_time
        
    c = a @ b
    torch.cuda.synchronize()  # wait for the GPU op to actually finish before looping
    iterations += 1

    _time = time.time()

 
elapsed = time.time() - start
print(f"Done: {iterations} matmuls in {elapsed:.1f}s "
      f"({iterations / elapsed:.1f} matmuls/sec)")

Using device: NVIDIA GeForce RTX 4070 Laptop GPU
Running repeated 6000x6000 matmuls on GPU for ~60 seconds...
Watch it with: watch -n 0.5 nvidia-smi   (in another terminal)


KeyboardInterrupt: 